# F0 — Hδ (4101.7 Å) και Paδ (10049 Å)

**Συμπλήρωμα εργαστηρίου φασματοσκοπίας.** Το παρόν περιέχει μόνο τα νέα κελιά που πρέπει να προστεθούν **στο υπάρχον notebook** *μετά* τον ορισμό `spectra`, `lines`, `measure_line`. Δεν αλλάζει την αρχική συνάρτηση του διδάσκοντος. Τα παράθυρα συνεχούς επιλέχθηκαν με οπτική επιθεώρηση των FITS.

Για **σύγκριση με μέλαν σώμα**, εκτέλεσε πρώτα στο αρχικό notebook τον ορισμό `plot_bb`, και μετά το τελευταίο κελί εδώ.

## A. Μετρήσεις με ακριβώς την αρχική μέθοδο

Πρόσθεσε το ακόλουθο στη θέση του παλαιού κελιού «Ερώτημα 2», όπου η `Pa7` ήταν σχολιασμένη.

In [ ]:
# F0: Οι δύο μετρήσεις EW — ίδια συνάρτηση του εργαστηρίου
F0_windows = {
    "Hd":  ([4050, 4058], [4160, 4169]),
    "Pa7": ([10020, 10029], [10094, 10100]),
}

EW_Hd = measure_line("F0", "Hd",
    cont_left=F0_windows["Hd"][0],
    cont_right=F0_windows["Hd"][1], plot=True)
EW_Pa7 = measure_line("F0", "Pa7",
    cont_left=F0_windows["Pa7"][0],
    cont_right=F0_windows["Pa7"][1], plot=True)

print(f"Hδ:  EW = {EW_Hd:.3f} Å")
print(f"Paδ: EW = {EW_Pa7:.3f} Å")

## B. Έλεγχος αβεβαιότητας EW

**Πρόσθετος διαγνωστικός έλεγχος**, όχι αλλαγή της `measure_line`. Χρησιμοποιεί **ακριβώς** τα ίδια παράθυρα, την ίδια ευθεία και τα ίδια όρια ολοκλήρωσης. Δίνει μία *τυπική* αβεβαιότητα από τον θόρυβο και την εκτίμηση της μέσης ροής των παραθύρων, υπό την απλοποιητική υπόθεση **ανεξάρτητων pixels**. Η πραγματική αβεβαιότητα μπορεί να είναι μεγαλύτερη λόγω συσχετίσεων και συστηματικών σφαλμάτων του συνεχούς.

In [ ]:
import numpy as np

def f0_ew_uncertainty(star, line, left, right, halfwidth=100):
    x_all, y_all = spectra[star]
    center = lines[line]["lam"]
    use = (x_all >= center-halfwidth) & (x_all <= center+halfwidth)
    x = np.asarray(x_all[use], dtype=float)
    y = np.asarray(y_all[use], dtype=float)

    l = (x > left[0]) & (x < left[1])
    r = (x > right[0]) & (x < right[1])
    w = l | r
    line_mask = (x > left[1]) & (x < right[0])
    if min(l.sum(), r.sum(), line_mask.sum()) < 2:
        raise ValueError("Πολύ λίγα σημεία στα παράθυρα / στην ολοκλήρωση")

    # ΙΔΙΟ συνεχές και ολοκλήρωμα με τη measure_line()
    xL, xR = np.mean(left), np.mean(right)
    yL, yR = y[l].mean(), y[r].mean()
    continuum = yL + (yR-yL)*(x-xL)/(xR-xL)
    dlambda = np.gradient(x)
    EW = np.sum((1-y[line_mask]/continuum[line_mask])*dlambda[line_mask])
    F = abs(np.sum((y[line_mask]-continuum[line_mask])*dlambda[line_mask]))
    sigma_pixel = np.std(y[w]-continuum[w], ddof=1)
    SN = F/(np.sqrt(line_mask.sum())*np.std(y[w]-continuum[w])*np.mean(dlambda[w]))

    # Τυπικό σφάλμα EW από τυχαίο θόρυβο στα pixels
    sigma_W_noise = np.sqrt(np.sum(
        (sigma_pixel*dlambda[line_mask]/continuum[line_mask])**2))

    # Επίδραση των αβεβαιοτήτων στη μέση ροή των ΔΥΟ παραθύρων
    alpha = (xR-x[line_mask])/(xR-xL)
    beta = 1-alpha
    dW_dyL = np.sum(y[line_mask]*dlambda[line_mask]*alpha / continuum[line_mask]**2)
    dW_dyR = np.sum(y[line_mask]*dlambda[line_mask]*beta / continuum[line_mask]**2)
    sigma_yL = np.std(y[l]-continuum[l],ddof=1)/np.sqrt(l.sum())
    sigma_yR = np.std(y[r]-continuum[r],ddof=1)/np.sqrt(r.sum())
    sigma_W_cont = np.hypot(dW_dyL*sigma_yL, dW_dyR*sigma_yR)
    sigma_W_formal = np.hypot(sigma_W_noise, sigma_W_cont)

    return dict(W=EW, F=F, SN=SN, N_left=int(l.sum()),
                N_right=int(r.sum()), sigma_noise=sigma_W_noise,
                sigma_cont=sigma_W_cont, sigma_formal=sigma_W_formal)

f0_Hd = f0_ew_uncertainty("F0", "Hd", *F0_windows["Hd"])
f0_Pa = f0_ew_uncertainty("F0", "Pa7", *F0_windows["Pa7"])
assert np.isclose(EW_Hd, f0_Hd["W"], atol=0.003)
assert np.isclose(EW_Pa7, f0_Pa["W"], atol=0.003)

for name, result in [("Hδ", f0_Hd), ("Paδ", f0_Pa)]:
    print(f'{name}: EW = {result["W"]:.3f} ± {result["sigma_formal"]:.3f} Å '
          f'(τυπική, χωρίς συστηματικά σφάλματα), S/N = {result["SN"]:.1f}, '
          f'N_cont = {result["N_left"]}+{result["N_right"]}')

### Προαιρετικός έλεγχος από τις στήλες FITS `PropErr` και `Std`

Το αρχικό notebook διαβάζει επιπλέον τη `PropErr` (προπαγόμενο σφάλμα ροής ανά δείγμα) και τη `Std` (διασπορά των προτύπων). Μπορούμε να υπολογίσουμε μια **δεύτερη** προσέγγιση της συνιστώσας θορύβου του EW από την `PropErr`. Δεν αντικαθιστά τον έλεγχο των παραθύρων ή τις μετρήσεις του εργαστηρίου. Υποθέτει πάλι ανεξάρτητα pixels.

In [ ]:
# Πρόσθετος έλεγχος: τυπική συνιστώσα σφάλματος από το FITS PropErr
from astropy.io import fits
import os

with fits.open(os.path.join(DATA_DIR, files["F0"])) as hdul:
    sp = hdul[1].data
    x_fits = 10**sp["LogLam"]
    y_fits = sp["Flux"]
    prop_err = sp["PropErr"]

for name, wl, win in [("Hδ", 4101.7, F0_windows["Hd"]),
                      ("Paδ", 10049.0, F0_windows["Pa7"])]:
    L, R = win
    m = (x_fits >= wl-100) & (x_fits <= wl+100)
    x, y, e = x_fits[m], y_fits[m], prop_err[m]
    left_mask = (x > L[0]) & (x < L[1])
    right_mask = (x > R[0]) & (x < R[1])
    line_mask = (x > L[1]) & (x < R[0])
    xL, xR = np.mean(L), np.mean(R)
    yL, yR = np.mean(y[left_mask]), np.mean(y[right_mask])
    continuum = yL + (yR-yL)*(x-xL)/(xR-xL)
    dlambda = np.gradient(x)
    if not np.all(np.isfinite(e[line_mask])):
        print(name, ": μη πεπερασμένες τιμές PropErr - απαιτείται έλεγχος")
        continue
    sigma_W_prop = np.sqrt(np.sum((e[line_mask]*dlambda[line_mask]
                                   /continuum[line_mask])**2))
    print(f"{name}: σ_EW από PropErr ≈ {sigma_W_prop:.3f} Å "
          "(μόνο ανεξάρτητος θόρυβος)")

## Γ. Ευαισθησία στην επιλογή των παραθύρων

**Αλλαγή παραθύρων ≠ στατιστικό σφάλμα.** Επαναλαμβάνουμε την **ίδια** μέθοδο για 4 κοντινές, οπτικά αποδεκτές επιλογές ανά γραμμή, ώστε να διαπιστώσουμε πόσο μεταβάλλεται το EW. Οι ελάχιστες και μέγιστες τιμές είναι *δοκιμή ευαισθησίας*, όχι 1σ διάστημα εμπιστοσύνης.

In [ ]:
# Εναλλακτικά κοντινά παράθυρα, μόνο για διαγνωστικό έλεγχο
alternatives = {
    "Hd": [
        ([4050,4058], [4160,4169]),
        ([4051,4057], [4161,4168]),
        ([4050,4059], [4160,4168]),
        ([4051,4058], [4160,4169]),
        ([4048,4058], [4160,4169]),
    ],
    "Pa7": [
        ([10020,10029], [10094,10100]),
        ([10021,10029], [10094,10099]),
        ([10020,10027], [10094,10100]),
        ([10022,10030], [10094,10100]),
        ([10024,10029], [10094,10100]),
    ],
}
W_ranges = {}
for line_name, choices in alternatives.items():
    values = [f0_ew_uncertainty("F0", line_name, L, R)["W"]
              for L, R in choices]
    W_ranges[line_name] = (min(values), max(values))
    print(f"{line_name}: κοντινές επιλογές -> EW από "
          f"{min(values):.3f} ως {max(values):.3f} Å")


## Δ. Θερμοκρασία διέγερσης (σύμφωνα με την §3.4 του φυλλαδίου)

**Προϋπόθεση**: και οι δύο γραμμές πρέπει να βρίσκονται στο γραμμικό μέρος της καμπύλης αύξησης. Αν δεν ισχύει, η τιμή που προκύπτει είναι μόνο **υπό συνθήκη εκτίμηση**.

\[
\frac{N_2}{N_3}=\frac{W_{H\delta}}{W_{Pa\delta}}\frac{f_{Pa\delta}}{f_{H\delta}}
\left(\frac{\lambda_{Pa\delta}}{\lambda_{H\delta}}\right)^2, \quad
T_\mathrm{exc}=\frac{E_3-E_2}{k_B\ln[(N_2/N_3)(g_3/g_2)]}.
\]

Οι τιμές ισχύος ταλαντωτή και το σχήμα του υπολογισμού προέρχονται από το φυλλάδιο, σελ. 27.

In [ ]:
# Φυσικές σταθερές και παράμετροι του οδηγού
lambda_Hd, lambda_Pa = 4101.7, 10049.0  # Å
osc_Hd, osc_Pa = 0.0221, 0.0277
kB_eV = 8.617333262e-5                  # eV/K
E2 = 13.6*(1-1/2**2)                    # eV
aE = 13.6*(1/2**2 - 1/3**2)              # E3-E2, eV
E3 = E2+aE
g2, g3 = 2*2**2, 2*3**2

N2_N3 = (EW_Hd/EW_Pa7)*(osc_Pa/osc_Hd)*(lambda_Pa/lambda_Hd)**2
Q = N2_N3*(g3/g2)
if Q <= 1:
    raise ValueError("Μη φυσικό αποτέλεσμα για θετική T — ελέγξτε τα EW")
T_exc = aE/(kB_eV*np.log(Q))

# Υπό ανεξάρτητες τυπικές αβεβαιότητες EW:
relN = np.hypot(f0_Hd["sigma_formal"]/EW_Hd,
                f0_Pa["sigma_formal"]/EW_Pa7)
sigma_ratio = N2_N3*relN
sigma_T_formal = T_exc/np.log(Q)*relN

print(f"E2={E2:.4f} eV, E3={E3:.4f} eV, ΔE={aE:.4f} eV")
print(f"g2={g2}, g3={g3}, N2/N3={N2_N3:.3f} ± {sigma_ratio:.3f} (τυπική)")
print(f"T_exc={T_exc:.0f} ± {sigma_T_formal:.0f} K (μόνο τυπικό σφάλμα)")
print(f"log10(W_Hδ/λ_Hδ)={np.log10(EW_Hd/lambda_Hd):.3f}")
print(f"log10(W_Paδ/λ_Paδ)={np.log10(EW_Pa7/lambda_Pa):.3f}")

# Εύρος T για συνδυασμούς των κοντινών εναλλακτικών παραθύρων
hmin,hmax = W_ranges["Hd"]
pmin,pmax = W_ranges["Pa7"]
def excitation_T(W_Hd,W_Pa):
    ratio=(W_Hd/W_Pa)*(osc_Pa/osc_Hd)*(lambda_Pa/lambda_Hd)**2
    return aE/(kB_eV*np.log(ratio*g3/g2))
t_sens = [excitation_T(h,p) for h in (hmin,hmax) for p in (pmin,pmax)]
print(f"Ευαισθησία παραθύρων: T ≈ {min(t_sens):.0f}–{max(t_sens):.0f} K")

# ΑΝΕΞΑΡΤΗΤΗ σύγκριση με θερμοκρασία αναφοράς, όχι fitting της θερμοκρασίας
print("T_eff (φυλλάδιο, F0): ~7400 K")
print("T_eff (FITs/λεξικό του οδηγού, εφόσον υπάρχει):",stars.get('F0'),"K")

## Ε. Σύγκριση με το μέλαν σώμα — προαιρετική

Αφού οριστεί η `plot_bb` στο **αρχικό notebook**, συγκρίνουμε το θερμικό συνεχές για την υπολογισμένη θερμοκρασία με εκείνο για τη θερμοκρασία αναφοράς. Το `T_exc` **δεν είναι** αναγκαστικά ίσο με `T_eff`, επειδή η πρώτη είναι τοπική θερμοκρασία διέγερσης και η δεύτερη ορίζεται από την ολική αστρική ροή.

In [ ]:
# Εκτέλεσέ το μόνο αφού εκτελέσεις το αρχικό κελί που ορίζει plot_bb
plot_bb("F0", T=T_exc)
plot_bb("F0", T=stars["F0"])

## Σημειώσεις για την εργασία (έως 2 σελίδες)

- Παρουσιάζουμε χωριστά τις δύο γραμμές με το επιλεγμένο συνεχές και τον πίνακα με W, S/N, τυπική αβεβαιότητα EW, f, λ, g, E.
- Η υπολογισμένη `T_exc` προκύπτει **υπό την υπόθεση γραμμικού τμήματος της καμπύλης αύξησης**. Οι πολύ ισχυρές γραμμές υδρογόνου (ιδίως η Hδ με μεγάλες πτέρυγες) είναι πιθανό να μην ικανοποιούν αυτή την υπόθεση. Δεν συνάγουμε χωρίς έλεγχο ότι η `T_exc` είναι η πραγματική `T_eff`.
- Η ηλιακή καμπύλη αύξησης Carroll–Ostlie, που παραπέμπει το φυλλάδιο, **δεν περιλαμβάνεται στα ανεβασμένα αρχεία**· χρειάζεται το συγκεκριμένο σχήμα για να τεκμηριώσουμε ποσοτικά σε ποιο τμήμα ανήκουν οι γραμμές.
- Οι μετρήσεις EW με ολοκλήρωση μεταξύ παραθύρων ενδέχεται να περιλαμβάνουν γειτονικές γραμμές. Ακριβέστερη απομόνωση του πραγματικού προφίλ (πυρήνας **και** πτέρυγες) είναι δυνητική βελτίωση. Στην παρούσα εργασία παραμένουμε στη μέθοδο του εργαστηρίου.
- Οι τυπικές αβεβαιότητες δεν καλύπτουν πλήρως τα συστηματικά σφάλματα επιλογής συνεχούς, τον κορεσμό και τις συσχετίσεις pixels.